# Etapa 5 · MariTalk (Sabiá-4) com prompting zero-shot

O Sabiá-4 (Maritaca AI) **não é treinado** neste trabalho: recebe instruções de corretor do ENEM e devolve um JSON com as 5 notas, justificativas e sugestões. Toda a lógica está em `src/maritalk.py`:

- validação do JSON (5 notas em {0, 40, …, 200}) e **total recalculado no código**;
- até 3 novas tentativas, espera crescente em erro 429, registro das falhas;
- **cache** de cada resposta válida (a mesma redação nunca é paga duas vezes);
- contagem de tokens e custo; trava de gasto máximo no lote.

> **Este notebook não faz chamadas pagas por padrão.** Ele lê os resultados já gerados em `results/`. Para chamar a API de novo, defina `RODAR_API = True` (requer `MARITACA_API_KEY` no `.env`).

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ / "src"))
import pandas as pd
import maritalk
from utils import COMPETENCIAS as C
RODAR_API = False
print("modelo:", maritalk.MODELO, "| prompt:", maritalk.VERSAO_PROMPT, "| temperatura:", maritalk.TEMPERATURA)

modelo: sabia-4 | prompt: v1 | temperatura: 0.0


## 5.1 O prompt (mensagem de sistema)

In [2]:
print(maritalk.SISTEMA)

Você é um corretor experiente da redação do ENEM e segue a matriz de referência do INEP.
Avalie a redação dissertativo-argumentativa abaixo nas 5 competências:

C1 - Domínio da modalidade escrita formal da língua portuguesa (gramática, ortografia, pontuação, registro).
C2 - Compreensão da proposta e aplicação de conceitos de várias áreas do conhecimento para desenvolver o tema,
     dentro dos limites estruturais do texto dissertativo-argumentativo em prosa.
C3 - Seleção, relação, organização e interpretação de informações, fatos, opiniões e argumentos em defesa de
     um ponto de vista (projeto de texto).
C4 - Conhecimento dos mecanismos linguísticos necessários para a construção da argumentação (coesão).
C5 - Elaboração de proposta de intervenção para o problema abordado, respeitando os direitos humanos
     (agente, ação, meio/modo, finalidade e detalhamento).

Níveis de cada competência:
200 = excelente/pleno; 160 = bom; 120 = mediano; 80 = insuficiente; 40 = precário; 0 = ausente

Cada redação é enviada na mensagem do usuário, junto com o título do tema:

In [3]:
print(maritalk.mensagem_usuario("<texto da redação>", "<título do tema>"))

Tema da proposta: <título do tema>

Redação:
<<<
<texto da redação>
>>>


## 5.2 Piloto (5 redações do **dev**) e estimativa de custo

O piloto usou redações do dev, não do teste, para que nenhuma decisão fosse tomada olhando o teste. Resultado registrado em 27/11/2026: 5/5 respostas válidas na 1ª tentativa; o modo JSON (`response_format={"type": "json_object"}`) foi aceito pela API; média de 976 tokens de entrada e 434 de saída; custo de R$ 0,0136 por redação → estimativa de R$ 13,35 para o teste. O prompt **não** foi ajustado depois do piloto (avaliação zero-shot).

In [4]:
if RODAR_API:
    dev = pd.read_csv(RAIZ / "data/dev.csv").sample(5, random_state=42)
    for i, r in enumerate(dev.itertuples()):
        out = maritalk.corrigir(r.texto, r.tema, json_mode=True)
        print(r.id, [out[c] for c in C], out["total"], "| real", r.score)
else:
    print("RODAR_API = False: piloto não executado de novo (resultados registrados acima).")

RODAR_API = False: piloto não executado de novo (resultados registrados acima).


## 5.3 Lote completo no teste

Executado pela linha de comando, com trava de gasto de R$ 16:

```
python src/maritalk.py --dados data/test.csv --limite-custo 16
```

In [5]:
if RODAR_API:
    maritalk.main(["--dados", str(RAIZ / "data/test.csv"), "--limite-custo", "16"])
resumo = json.loads((RAIZ / "results/maritalk_resumo.json").read_text(encoding="utf-8"))
pd.Series(resumo)

n                                                      985
ok                                                     985
falhas                                                   0
taxa_falha                                             0.0
chamadas_novas                                         985
tokens_entrada                                      992549
tokens_saida                                        433188
custo_chamadas_novas_rs                            13.6265
custo_estimado_rs                                  13.6265
interrompido_por_limite                              False
modelo                                             sabia-4
versao_prompt                                           v1
temperatura                                            0.0
preco_rs_por_milhao        {'entrada': 5.0, 'saida': 20.0}
dtype: object

In [6]:
falhas = pd.read_csv(RAIZ / "results/falhas_maritalk.csv")
preds = pd.read_csv(RAIZ / "results/preds_maritalk.csv")
test = pd.read_csv(RAIZ / "data/test.csv")
print(f"previsões válidas: {len(preds)} de {len(test)} | falhas: {len(falhas)} "
      f"| taxa de falha: {resumo['taxa_falha']:.1%}")
print(f"custo real: R$ {resumo['custo_estimado_rs']:.2f} "
      f"(R$ {resumo['custo_estimado_rs'] / max(resumo['ok'], 1):.4f} por redação)")

previsões válidas: 985 de 985 | falhas: 0 | taxa de falha: 0.0%
custo real: R$ 13.63 (R$ 0.0138 por redação)


## 5.4 Distribuição das notas atribuídas (comparada com as reais)

In [7]:
dist = pd.concat({
    "real": test[C].apply(lambda s: s.value_counts(normalize=True)),
    "MariTalk": preds[C].apply(lambda s: s.value_counts(normalize=True)),
}, axis=1).reindex([0, 40, 80, 120, 160, 200]).fillna(0)
(dist * 100).round(1)

real                         MariTalk                        
       c1    c2    c3    c4    c5       c1    c2    c3    c4    c5
0     1.2   1.7   2.9   3.2   7.9      0.1   0.2   0.2   0.1  18.3
40    0.3   1.8   2.8   1.0   4.4     10.9   0.6   1.3   3.7   4.9
80    7.9  12.5  22.6  11.9  19.6     56.1  18.5  37.7  51.8  25.6
120  46.8  36.8  46.6  38.3  33.9     24.5  46.4  48.9  35.8  37.3
160  39.1  38.0  23.0  27.4  24.2      8.4  32.9  11.9   8.6  13.5
200   4.7   9.2   1.9  18.2  10.1      0.0   1.4   0.0   0.0   0.5

In [8]:
media = pd.DataFrame({"real (média)": test.set_index("id").loc[preds.id, C + ["score"]].mean().values,
                      "MariTalk (média)": preds[C + ["total"]].mean().values},
                     index=C + ["total"]).round(1)
media["diferença"] = media["MariTalk (média)"] - media["real (média)"]
media

,real (média),MariTalk (média),diferença
c1,134.5,92.1,-42.4
c2,134.1,126.2,-7.9
c3,115.9,108.4,-7.5
c4,136.0,99.7,-36.3
c5,116.9,89.7,-27.2
total,637.4,516.1,-121.3


O MariTalk é **sistematicamente mais severo** que os corretores do corpus, sobretudo em C1 (norma padrão) e C4 (coesão). As métricas completas e a comparação com os outros modelos estão em `results/tabela_final.md` (gerada por `src/avaliacao.py`).

## 5.5 Exemplo de justificativas e sugestões

In [9]:
just = [json.loads(l) for l in open(RAIZ / "results/maritalk_justificativas.jsonl", encoding="utf-8")]
ex = just[0]
linha = preds.set_index("id").loc[ex["id"]]
real = test.set_index("id").loc[ex["id"]]
print(f"{ex['id']} | tema: {real['tema']}")
print("MariTalk:", [int(linha[c]) for c in C], "=", int(linha["total"]), "| real:", [int(real[c]) for c in C], "=", int(real["score"]))
for c in C:
    print(f"\n{c.upper()}: {ex['justificativas'][c]}")
print("\nSugestões:")
for s in ex["sugestoes"]:
    print(" -", s)

essay_00697 | tema: Desmatamento e queimadas: a importância de preservar a floresta amazônica
MariTalk: [80, 120, 80, 80, 80] = 440 | real: [120, 80, 80, 120, 40] = 440

C1: Há diversos erros de ortografia, acentuação e concordância que prejudicam a correção gramatical do texto.

C2: O tema é abordado de forma geral, com menção a causas e consequências, mas faltam aprofundamento e repertório sociocultural diversificado.

C3: A organização das ideias é fraca, com argumentos apresentados de forma superficial e sem encadeamento lógico claro.

C4: A coesão é insuficiente: há frases soltas, repetição de ideias e conectivos escassos entre os parágrafos.

C5: A proposta de intervenção existe, mas é genérica, com poucos detalhes sobre agentes, meios e modo de execução.

Sugestões:
 - Revisar ortografia, acentuação e concordância antes de entregar a redação.
 - Estruturar melhor o texto em introdução, desenvolvimento e conclusão, aprofundando causas, consequências e soluções.
 - Detalhar a prop